In [2]:
!pip install -U transformers peft bitsandbytes accelerate pillow tqdm nltk rouge-score bert-score 
!pip install gdown
!pip install huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 316.6 kB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 371.5 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 450.4 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 486.6 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 501.8 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 1.5 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 3.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 21.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 24.0 MB/s eta 0:0

In [10]:
!pip install -U bitsandbytes>=0.46.1
!pip install peft --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 8.8 MB/s eta 0:00:00a 0:00:01


In [8]:
import shutil

In [2]:
!gdown --fuzzy "https://drive.google.com/file/d/1UVmg3sLs9hS-5TMqm6f-j8TOvyHsqFIq/view?usp=drive_link" -O model.zip

Downloading...
From (original): https://drive.google.com/uc?id=1UVmg3sLs9hS-5TMqm6f-j8TOvyHsqFIq
From (redirected): https://drive.google.com/uc?id=1UVmg3sLs9hS-5TMqm6f-j8TOvyHsqFIq&confirm=t&uuid=727fd3c6-9c72-4f18-b16e-9304b6b53145
To: /workspace/model.zip
100%|████████████████████████████████████████| 852M/852M [00:22<00:00, 38.2MB/s]


In [3]:
# Unpack entire archive
shutil.unpack_archive('model.zip', 'model')

In [ ]:
import os
import json
import math
import gc
import zipfile
import traceback
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import torch, gc
from tqdm import tqdm
from PIL import Image

from transformers import AutoProcessor, BitsAndBytesConfig
from transformers import Gemma3ForConditionalGeneration
from peft import PeftModel
from huggingface_hub import login

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score as bertscore_score

# Login to Hugging Face
login(token="")

In [30]:
def safe_mean(xs: List[float]) -> float:
    return float(sum(xs) / len(xs)) if xs else 0.0


class VLMBenchmarkAdapted:
    """
    Benchmark for adapted Gemma3 VQA (base + LoRA adapter already loaded outside).
    - Generates answers with processor.apply_chat_template
    - Computes per-QA: loss, perplexity (on generated answer tokens only), BLEU (word-level), ROUGE (optional)
    - Computes BERTScore in batch at end via run_bert()
    - Checkpoint/resume per image id
    """

    def __init__(
        self,
        processor: Any,
        model: torch.nn.Module,
        device: torch.device,
        checkpoint_file: str = "checkpoint.json",
        bertscore_model_type: str = "bert-base-multilingual-cased",
        bertscore_lang: str = "si",
    ):
        print("Initializing benchmark (adapted model)...")

        self.processor = processor
        self.model = model
        self.device = device
        self.checkpoint_file = checkpoint_file


        self.bertscore_model_type = bertscore_model_type
        self.bertscore_lang = bertscore_lang

        self.results: List[Dict] = []
        self.errors: List[Dict] = []
        self.processed_ids: set = set()

        self._bertscore_done: bool = False

        print("✓ Benchmark initialized")

    # -----------------------------
    # Checkpointing
    # -----------------------------
    def load_checkpoint(self) -> bool:
        if os.path.exists(self.checkpoint_file):
            print(f"Loading checkpoint from {self.checkpoint_file}...")
            try:
                with open(self.checkpoint_file, "r", encoding="utf-8") as f:
                    checkpoint = json.load(f)

                self.results = checkpoint.get("results", [])
                self.errors = checkpoint.get("errors", [])
                self.processed_ids = set(checkpoint.get("processed_ids", []))

                # detect if bertscore already exists
                self._bertscore_done = bool(checkpoint.get("bertscore_done", False))

                print(f"✓ Resumed: {len(self.results)} images, {len(self.errors)} errors")
                return True
            except Exception as e:
                print(f"Error loading checkpoint: {e}")
                return False
        return False

    def save_checkpoint(self) -> None:
        try:
            checkpoint = {
                "results": self.results,
                "errors": self.errors,
                "processed_ids": list(self.processed_ids),
                "bertscore_done": self._bertscore_done,
            }
            with open(self.checkpoint_file, "w", encoding="utf-8") as f:
                json.dump(checkpoint, f, ensure_ascii=False, indent=2)
        except Exception as e:
            print(f"Error saving checkpoint: {e}")

    # -----------------------------
    # Data utilities
    # -----------------------------
    def unzip_images(self, zip_path: str, extract_to: str) -> None:
        print(f"Unzipping {zip_path} to {extract_to}...")
        if os.path.exists(extract_to):
            print(f"✓ Dataset folder already exists at {extract_to}")
            return
        try:
            with zipfile.ZipFile(zip_path, "r") as zip_ref:
                zip_ref.extractall(extract_to)
            print(f"✓ Successfully extracted to {extract_to}")
        except Exception as e:
            print(f"Error unzipping: {e}")
            raise

    def load_json(self, json_path: str) -> List[Dict]:
        print(f"Loading JSON from {json_path}...")
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"✓ Loaded {len(data)} items")
        return data

    def find_image(self, image_id: int, dataset_folder: str) -> str:
        extensions = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]
        # Try root first
        for ext in extensions:
            p = os.path.join(dataset_folder, f"{image_id}{ext}")
            if os.path.exists(p):
                return p
        # Walk subfolders
        for root, _, _files in os.walk(dataset_folder):
            for ext in extensions:
                p = os.path.join(root, f"{image_id}{ext}")
                if os.path.exists(p):
                    return p
        raise FileNotFoundError(f"Image with ID {image_id} not found in {dataset_folder}")

    # -----------------------------
    # Metrics: BLEU + ROUGE
    # -----------------------------
    def calculate_bleu(self, reference: str, hypothesis: str) -> float:
        """
        BLEU for Sinhala (word-level). If you want char-level:
        ref_tokens = list(reference.strip()); hyp_tokens = list(hypothesis.strip())
        """
        try:
            ref_tokens = (reference or "").strip().split()
            hyp_tokens = (hypothesis or "").strip().split()
            smoothing = SmoothingFunction().method1
            bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smoothing)
            return float(bleu * 100.0)
        except Exception as e:
            print(f"Error calculating BLEU: {e}")
            return 0.0

    # -----------------------------
    # Core: inference + loss/ppl on generated answer tokens only
    # -----------------------------
    @torch.inference_mode()
    def _generate_answer_and_nll(
        self,
        image: Image.Image,
        question: str,
        max_new_tokens: int = 128,
    ) -> Tuple[str, Optional[float], Optional[float], int, int]:
    
        messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": "ඔබ ප්‍රශ්නවලට නිවැරදිව පිළිතුරු දෙන උපකාරකයෙකි."}],
            },
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": f"මෙම රූපය බලා පහත ප්‍රශ්නයට පිළිතුරු දෙන්න:\nප්‍රශ්නය: {question}"},
                ],
            },
        ]
    
        # ── Tokenise — keep int tensors, only move device ──
        inputs = self.processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            padding=True,
        )
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
    
        input_len = int(inputs["input_ids"].shape[-1])
    
        # ── Generate ──
        gen = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    
        answer_ids  = gen[0][input_len:]
        answer_text = self.processor.decode(answer_ids, skip_special_tokens=True).strip()
        answer_len  = int(answer_ids.shape[0])
    
        if answer_len == 0:
            return answer_text, None, None, input_len, 0
    
        # ── NLL over generated tokens only ──
        full_ids       = gen                                          # [1, seq]
        attention_mask = torch.ones_like(full_ids)
        # token_type_ids required by Gemma3 — all 0 (no image tokens in the full generated sequence)
        token_type_ids = torch.zeros_like(full_ids)
    
        labels = full_ids.clone()
        labels[:, :input_len] = -100   # mask prompt, compute loss on answer only
    
        outputs = self.model(
            input_ids=full_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels,
        )
    
        loss = outputs.loss.item() if outputs.loss is not None else None
        ppl  = float(math.exp(loss)) if loss is not None else None
    
        return answer_text, loss, ppl, input_len, answer_len
    

    def process_single_qa(self, image_path, question, ground_truth, qa_id, image_id, max_new_tokens=128):
        try:
            img = Image.open(image_path).convert("RGB")
            answer_text, loss, ppl, input_len, ans_len = self._generate_answer_and_nll(img, question, max_new_tokens)
            bleu = self.calculate_bleu(ground_truth, answer_text)
    
            return {
                "qa_id": qa_id,
                "id": image_id,
                "question": question,
                "ground_truth": ground_truth,
                "answer": answer_text,
                "loss": loss,
                "perplexity": ppl,
                "bleu": bleu,
                "bertscore": None,
                "input_len_tokens": input_len,
                "answer_len_tokens": ans_len,
                "status": "success",
            }
        except Exception as e:
            error_msg = f"{type(e).__name__}: {str(e)}"
            print(f"Error processing QA {qa_id} (image {image_id}): {error_msg}")
            print(traceback.format_exc())

            self.errors.append(
                {
                    "qa_id": qa_id,
                    "id": image_id,
                    "error": error_msg,
                    "traceback": traceback.format_exc(),
                }
            )
            return {
                "qa_id": qa_id,
                "id": image_id,
                "question": question,
                "ground_truth": ground_truth,
                "answer": None,
                "loss": None,
                "perplexity": None,
                "bleu": None,
                "bertscore": None,
                "status": "error",
                "error": error_msg,
            }

    def process_dataset(
        self,
        json_path: str,
        dataset_folder: str,
        save_every_images: int = 10,
        max_new_tokens: int = 128,
    ) -> None:
        data = self.load_json(json_path)

        items_to_process = [item for item in data if item["id"] not in self.processed_ids]
        if len(items_to_process) < len(data):
            print(f"Skipping {len(data) - len(items_to_process)} already processed images")

        for idx, item in enumerate(tqdm(items_to_process, desc="Processing images")):
            image_id = item["id"]
            try:
                image_path = self.find_image(image_id, dataset_folder)

                qas_out = []
                for qa in tqdm(item["qas"], desc=f"Image {image_id}", leave=False):
                    qas_out.append(
                        self.process_single_qa(
                            image_path=image_path,
                            question=qa["question"],
                            ground_truth=qa["answer"],
                            qa_id=qa["qa_id"],
                            image_id=image_id,
                            max_new_tokens=max_new_tokens,
                        )
                    )
                    torch.cuda.empty_cache()
                    gc.collect()

                self.results.append({"id": image_id, "qas": qas_out})
                self.processed_ids.add(image_id)

                if (idx + 1) % save_every_images == 0:
                    self.save_checkpoint()

            except Exception as e:
                error_msg = f"{type(e).__name__}: {str(e)}"
                print(f"Error processing image {image_id}: {error_msg}")
                print(traceback.format_exc())

                for qa in item.get("qas", []):
                    self.errors.append({"qa_id": qa.get("qa_id"), "id": image_id, "error": error_msg})

                self.processed_ids.add(image_id)
                self.save_checkpoint()

        self.save_checkpoint()

    # -----------------------------
    # Batch BERTScore
    # -----------------------------
    def run_bert(self, batch_size: int = 32, overwrite: bool = False) -> Dict[str, float]:
        """
        Compute BERTScore for all successful QAs in self.results in batches.
        Writes per-QA bertscore back into results.
        If overwrite=False, skips QAs that already have bertscore.
        """
        refs: List[str] = []
        hyps: List[str] = []
        index_map: List[Tuple[int, int]] = []

        for item_idx, item in enumerate(self.results):
            for qa_idx, qa in enumerate(item.get("qas", [])):
                if qa.get("status") != "success":
                    continue
                if qa.get("answer") is None:
                    continue
                if (not overwrite) and (qa.get("bertscore") is not None):
                    continue

                refs.append(qa.get("ground_truth", ""))
                hyps.append(qa.get("answer", ""))
                index_map.append((item_idx, qa_idx))

        if not hyps:
            print("No QAs found for BERTScore (maybe already computed).")
            self._bertscore_done = True
            self.save_checkpoint()
            return {
                "mean_bertscore_precision": 0.0,
                "mean_bertscore_recall": 0.0,
                "mean_bertscore_f1": 0.0,
            }

        print(f"Running BERTScore on {len(hyps)} QAs (batch_size={batch_size})...")

        P, R, F1 = bertscore_score(
            cands=hyps,
            refs=refs,
            model_type=self.bertscore_model_type,
            lang=self.bertscore_lang,
            device=str(self.device),
            batch_size=batch_size,
            verbose=True,
        )

        for i, (item_idx, qa_idx) in enumerate(index_map):
            self.results[item_idx]["qas"][qa_idx]["bertscore"] = {
                "bertscore_precision": float(P[i].item() * 100.0),
                "bertscore_recall": float(R[i].item() * 100.0),
                "bertscore_f1": float(F1[i].item() * 100.0),
            }

        self._bertscore_done = True
        self.save_checkpoint()

        agg = {
            "mean_bertscore_precision": float(P.mean().item() * 100.0),
            "mean_bertscore_recall": float(R.mean().item() * 100.0),
            "mean_bertscore_f1": float(F1.mean().item() * 100.0),
        }
        print("✓ BERTScore done:", agg)
        return agg

    # -----------------------------
    # Aggregate metrics
    # -----------------------------
    def calculate_aggregate_metrics(self) -> Dict[str, float]:
        bleus, losses, ppls = [], [], []
        r1, r2, rL = [], [], []
        bP, bR, bF = [], [], []
        total_qas, success_qas = 0, 0

        for item in self.results:
            for qa in item["qas"]:
                total_qas += 1
                if qa.get("status") != "success":
                    continue
                success_qas += 1

                if qa.get("bleu") is not None:
                    bleus.append(qa["bleu"])
                if qa.get("loss") is not None:
                    losses.append(qa["loss"])
                if qa.get("perplexity") is not None:
                    ppls.append(qa["perplexity"])

                if qa.get("bertscore") is not None:
                    bP.append(qa["bertscore"]["bertscore_precision"])
                    bR.append(qa["bertscore"]["bertscore_recall"])
                    bF.append(qa["bertscore"]["bertscore_f1"])

        return {
            "mean_bleu": safe_mean(bleus),
            "mean_loss": safe_mean(losses),
            "mean_perplexity": safe_mean(ppls),
            "mean_bertscore_precision": safe_mean(bP),
            "mean_bertscore_recall": safe_mean(bR),
            "mean_bertscore_f1": safe_mean(bF),
            "total_qas": float(total_qas),
            "success_qas": float(success_qas),
            "success_rate": float(success_qas / total_qas * 100.0) if total_qas else 0.0,
        }

    # -----------------------------
    # Save
    # -----------------------------
    def save_results(self, output_path: str) -> None:
        print(f"Saving results to {output_path}...")
        agg = self.calculate_aggregate_metrics()
        out = {
            "results": self.results,
            "aggregate_metrics": agg,
            "errors": self.errors,
        }
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(out, f, ensure_ascii=False, indent=2)

        print("✓ Saved")
        print("\n=== Aggregate Metrics ===")
        for k, v in agg.items():
            if isinstance(v, float):
                print(f"{k}: {v:.4f}")
            else:
                print(f"{k}: {v}")

    def save_errors(self, error_path: str) -> None:
        if not self.errors:
            print("No errors to save!")
            return
        print(f"Saving {len(self.errors)} errors to {error_path}...")
        with open(error_path, "w", encoding="utf-8") as f:
            json.dump(self.errors, f, ensure_ascii=False, indent=2)
        print("✓ Errors saved")

print("Adapted benchmark class defined")

Adapted benchmark class defined


In [31]:
BASE_MODEL = "google/gemma-3-4b-it"
ADAPTER_DIR = "mixed_output_20260309_091408/final_adapter"  # <-- your adapter folder
TEST_JSON = "data/test_1000_test.json"  # <-- your test set json
IMAGES_DIR = "data/filtered_images"               # <-- folder containing images (or root with subfolders)

OUT_RESULTS = "adapted_mixed_test_results-not-pt.json"
OUT_ERRORS = "adapted_mixed_test_errors-not-pt.json"
CHECKPOINT = "adapted_mixed_checkpoint-not-pt.json"

MAX_NEW_TOKENS = 128
SAVE_EVERY_IMAGES = 10

In [18]:

# -----------------------------
# Load model
# -----------------------------
torch.cuda.empty_cache()
gc.collect()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading base model (4-bit)...")
base_model = Gemma3ForConditionalGeneration.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
).eval()

print("Loading processor from adapter dir...")
processor = AutoProcessor.from_pretrained(ADAPTER_DIR)

print("Attaching LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR).eval()

device = model.device  # usually cuda:0 (device_map may spread, but generate uses correct placement)

print("model loaded...")


Loading base model (4-bit)...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Loading processor from adapter dir...
Attaching LoRA adapter...
model loaded...


In [32]:
# ================================
# run_benchmark_adapted.py (UPDATED)
# ================================
bench = VLMBenchmarkAdapted(
    processor=processor,
    model=model,
    device=device,
    checkpoint_file=CHECKPOINT,
    bertscore_model_type="bert-base-multilingual-cased",
    bertscore_lang="si",
)

bench.load_checkpoint()

bench.process_dataset(
    json_path=TEST_JSON,
    dataset_folder=IMAGES_DIR,
    save_every_images=SAVE_EVERY_IMAGES,
    max_new_tokens=MAX_NEW_TOKENS,
)

# NEW: batch bertscore after generation
bench.run_bert(batch_size=32)

bench.save_results(OUT_RESULTS)
bench.save_errors(OUT_ERRORS)
print("✓ Done")


Initializing benchmark (adapted model)...
✓ Benchmark initialized
Loading JSON from data/test_1000_test.json...
✓ Loaded 910 items


Processing images:  62%|██████▏   | 564/910 [36:23<20:19,  3.52s/it]

Error processing image 2414497: FileNotFoundError: Image with ID 2414497 not found in data/filtered_images
Traceback (most recent call last):
  File "/tmp/ipykernel_10051/1399412533.py", line 272, in process_dataset
    image_path = self.find_image(image_id, dataset_folder)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_10051/1399412533.py", line 115, in find_image
    raise FileNotFoundError(f"Image with ID {image_id} not found in {dataset_folder}")
FileNotFoundError: Image with ID 2414497 not found in data/filtered_images

Error processing image 2404556: FileNotFoundError: Image with ID 2404556 not found in data/filtered_images
Traceback (most recent call last):
  File "/tmp/ipykernel_10051/1399412533.py", line 272, in process_dataset
    image_path = self.find_image(image_id, dataset_folder)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_10051/1399412533.py", line 115, in find_image
    raise FileNotFoundError(f"I


Processing images: 100%|██████████| 910/910 [1:00:06<00:00,  3.96s/it]


Running BERTScore on 996 QAs (batch_size=32)...


...

calculating scores...
computing bert embedding.


  0%|          | 0/58 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/32 [00:00<?, ?it/s]

done in 0.80 seconds, 1249.97 sentences/sec
✓ BERTScore done: {'mean_bertscore_precision': 82.46148228645325, 'mean_bertscore_recall': 86.13390922546387, 'mean_bertscore_f1': 84.1977059841156}
Saving results to adapted_mixed_test_results-not-pt.json...
✓ Saved

=== Aggregate Metrics ===
mean_bleu: 1.2062
mean_loss: 0.9443
mean_perplexity: 56.1732
mean_bertscore_precision: 82.4615
mean_bertscore_recall: 86.1339
mean_bertscore_f1: 84.1977
total_qas: 996.0000
success_qas: 996.0000
success_rate: 100.0000
Saving 4 errors to adapted_mixed_test_errors-not-pt.json...
✓ Errors saved
✓ Done


In [14]:
import os
import zipfile

def zip_folder(folder_path, zip_path):
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, folder_path)
                zipf.write(file_path, arcname)

zip_folder("mixed_output_20260309_091408", "mixed_cpt_vqa.zip")